Cấu hình hệ thống và Quản lý luồng (Pipeline Setup)

In [1]:
pip install mlxtend

In [2]:
import pandas as pd
import numpy as np
import os
import gc
from mlxtend.frequent_patterns import fpgrowth, association_rules

# 1. Cấu hình bản đồ đường dẫn tích hợp cho toàn bộ Giai đoạn 2
file_pipeline_configs = [
    {
        "input_cleaned": "2019-Oct-Cleaned.csv",
        "output_cube": "olap_cube/oct_star_cube.csv",
        "output_iceberg": "iceberg/oct_iceberg_cube.csv",
        "output_rules": "rules/oct_association_rules.csv"
    },
    {
        "input_cleaned": "2019-Nov-Cleaned.csv",
        "output_cube": "olap_cube/nov_star_cube.csv",
        "output_iceberg": "iceberg/nov_iceberg_cube.csv",
        "output_rules": "rules/nov_association_rules.csv"
    }
]

# CHỈ CHỈNH SỬA TẠI ĐÂY: 
# Đặt = 0 để chạy toàn bộ quy trình cho file Oct
# Đặt = 1 để chạy toàn bộ quy trình cho file Nov
CURRENT_FILE_INDEX = 0 

# 2. Trích xuất cấu hình mục tiêu
current_config = file_pipeline_configs[CURRENT_FILE_INDEX]
PATH_INPUT = current_config["input_cleaned"]
PATH_CUBE = current_config["output_cube"]
PATH_ICEBERG = current_config["output_iceberg"]
PATH_RULES = current_config["output_rules"]

# 3. Khởi tạo cấu trúc thư mục lưu trữ vật lý cho DWH và Data Mining
os.makedirs('olap_cube', exist_ok=True)
os.makedirs('iceberg', exist_ok=True)
os.makedirs('rules', exist_ok=True)

print(f"=== PIPELINE READY ===")
print(f"[*] Target Input: {PATH_INPUT}")
print(f"[*] Target DWH Cube: {PATH_CUBE}")
print(f"[*] Target Iceberg: {PATH_ICEBERG}")
print(f"[*] Target Rules: {PATH_RULES}")

=== PIPELINE READY ===
[*] Target Input: 2019-Oct-Cleaned.csv
[*] Target DWH Cube: olap_cube/oct_star_cube.csv
[*] Target Iceberg: iceberg/oct_iceberg_cube.csv
[*] Target Rules: rules/oct_association_rules.csv


Trích xuất & Thiết lập Chiều Thời gian (ETL - Time Dimension)

In [3]:
print(f"[ETL - Step 1] Đang trích xuất dữ liệu từ nguồn sạch: {PATH_INPUT}...")
df = pd.read_csv(PATH_INPUT)

print("[ETL - Step 2] Đang đồng bộ hóa định dạng Datetime hệ thống...")
df['event_time'] = pd.to_datetime(df['event_time'])

print("[ETL - Step 3] Xây dựng phân cấp thuộc tính cho Chiều Thời gian (Time Dimension Hierarchy)...")
# Khởi tạo các tầng cấp độ từ Chi tiết (Giờ) đến Tổng quát (Tháng)
df['hour'] = df['event_time'].dt.hour
df['day'] = df['event_time'].dt.day
df['day_of_week'] = df['event_time'].dt.day_name()
df['month'] = df['event_time'].dt.month

print(f"-> Hoàn tất cấu trúc Dim_Time. Kích thước bản ghi ghi nhận: {df.shape}")
df[['event_time', 'hour', 'day', 'day_of_week', 'month']].head(3)

[ETL - Step 1] Đang trích xuất dữ liệu từ nguồn sạch: 2019-Oct-Cleaned.csv...
[ETL - Step 2] Đang đồng bộ hóa định dạng Datetime hệ thống...
[ETL - Step 3] Xây dựng phân cấp thuộc tính cho Chiều Thời gian (Time Dimension Hierarchy)...
-> Hoàn tất cấu trúc Dim_Time. Kích thước bản ghi ghi nhận: (42418544, 13)


,event_time,hour,day,day_of_week,month
0,2019-10-01 00:00:00+00:00,0,1,Tuesday,10
1,2019-10-01 00:00:00+00:00,0,1,Tuesday,10
2,2019-10-01 00:00:01+00:00,0,1,Tuesday,10


Tổng hợp Đa chiều & Khởi tạo Kho dữ liệu Star Schema (ETL - Load Cube)

In [4]:
print("[ETL - Step 4] Khởi tạo cấu hình Sơ đồ hình sao (Star Schema)...")
# Định nghĩa các trục Dimension phẳng kết nối trực tiếp vào Fact
dimensions_axes = ['brand', 'category_code', 'day_of_week', 'hour']

print("[ETL - Step 5] Tính toán các đại lượng đo (Measures) và nạp dữ liệu vào Data Cube...")
# Xây dựng các cột bổ trợ tối ưu tốc độ tính toán
df['is_purchase'] = (df['event_type'] == 'purchase').astype(int)
df['revenue_measure'] = np.where(df['event_type'] == 'purchase', df['price'], 0)

# Thực hiện gom nhóm đa chiều để nạp vào Fact Table của Data Warehouse
data_cube = df.groupby(dimensions_axes).agg(
    total_interactions=('price', 'count'),
    total_purchases=('is_purchase', 'sum'),
    total_revenue=('revenue_measure', 'sum')
).reset_index()

print(f"[ETL - Step 6] Nạp dữ liệu vật lý vào Kho lưu trữ OLAP Cube...")
data_cube.to_csv(PATH_CUBE, index=False)
print(f" -> Xuất Base Cuboid thành công: {PATH_CUBE}")

# BƯỚC TỐI ƯU: Trích xuất sớm tập giao dịch cho Cell 6 khi df còn sống
print("[Tối ưu RAM] Đang trích xuất trước tập giỏ hàng cho thuật toán FP-Growth...")
df_basket_source = df[['user_session', 'brand']].dropna()
df_basket_source = df_basket_source[df_basket_source['brand'] != 'Unknown'].head(600000).copy()

# Xóa sạch df thô ra khỏi RAM
del df
gc.collect()
data_cube.head(5)

[ETL - Step 4] Khởi tạo cấu hình Sơ đồ hình sao (Star Schema)...
[ETL - Step 5] Tính toán các đại lượng đo (Measures) và nạp dữ liệu vào Data Cube...
[ETL - Step 6] Nạp dữ liệu vật lý vào Kho lưu trữ OLAP Cube...
 -> Xuất Base Cuboid thành công: olap_cube/oct_star_cube.csv
[Tối ưu RAM] Đang trích xuất trước tập giỏ hàng cho thuật toán FP-Growth...


,brand,category_code,day_of_week,hour,total_interactions,total_purchases,total_revenue
0,Unknown,Unknown,Friday,0,4022,28,5194.95
1,Unknown,Unknown,Friday,1,6503,45,5175.81
2,Unknown,Unknown,Friday,2,12052,115,16253.06
3,Unknown,Unknown,Friday,3,18043,227,30049.74
4,Unknown,Unknown,Friday,4,21086,289,34938.54


Thực thi các Thao tác Phân tích Kho dữ liệu (OLAP Operations)

In [5]:
print("=== THỰC THI ENGINE PHÂN TÍCH ĐA CHIỀU OLAP ===")

# Thao tác 1: SLICE (Cắt dữ liệu phẳng theo một điều kiện chiều duy nhất)
print("\n[OLAP Operation 1] SLICE - Lọc phân hệ doanh thu riêng cho thương hiệu 'apple':")
slice_df = data_cube[data_cube['brand'] == 'apple']
print(slice_df.head(3))

# Thao tác 2: DICE (Xúc xắc khối - Lọc tiểu khối theo nhiều chiều đồng thời)
print("\n[OLAP Operation 2] DICE - Lọc phân khúc Samsung, ngành Smartphone, khung giờ đêm (20h - 23h):")
dice_df = data_cube[
    (data_cube['brand'] == 'samsung') & 
    (data_cube['category_code'].str.contains('smartphone', na=False)) & 
    (data_cube['hour'].between(20, 23))
]
print(dice_df.head(3))

# Thao tác 3: ROLL-UP (Tổng hợp ngược lên - Thu hẹp chiều chi tiết để tăng tính tổng quát)
print("\n[OLAP Operation 3] ROLL-UP - Loại bỏ yếu tố thời gian, tổng hợp vĩ mô theo Brand và Category:")
rollup_df = data_cube.groupby(['brand', 'category_code'])[['total_interactions', 'total_purchases', 'total_revenue']].sum().reset_index()
print(rollup_df.sort_values(by='total_revenue', ascending=False).head(3))

# Thao tác 4: PIVOT (Xoay khối - Đổi trục biểu diễn để so sánh chéo trực quan)
print("\n[OLAP Operation 4] PIVOT - Ma trận xoay phân bổ Doanh thu giữa Thương hiệu và Thứ trong tuần:")
pivot_matrix = data_cube.pivot_table(
    values='total_revenue', 
    index='brand', 
    columns='day_of_week', 
    aggfunc='sum'
).fillna(0)
print(pivot_matrix.head(5))

=== THỰC THI ENGINE PHÂN TÍCH ĐA CHIỀU OLAP ===

[OLAP Operation 1] SLICE - Lọc phân hệ doanh thu riêng cho thương hiệu 'apple':
       brand category_code day_of_week  hour  total_interactions  \
42477  apple       Unknown      Friday     0                  33   
42478  apple       Unknown      Friday     1                  30   
42479  apple       Unknown      Friday     2                 116   

       total_purchases  total_revenue  
42477                0            0.0  
42478                0            0.0  
42479                0            0.0  

[OLAP Operation 2] DICE - Lọc phân khúc Samsung, ngành Smartphone, khung giờ đêm (20h - 23h):
          brand           category_code day_of_week  hour  total_interactions  \
458631  samsung  electronics.smartphone      Friday    20                7935   
458632  samsung  electronics.smartphone      Friday    21                4625   
458633  samsung  electronics.smartphone      Friday    22                2752   

        total_purc

Lập trình Thuật toán tính toán Iceberg Cube

In [6]:
print("=== THỰC THI THUẬT TOÁN ICEBERG CUBE TỐI ƯU ===")

# 1. Thiết lập hệ thống ngưỡng điều kiện hạ tầng cho Tảng Băng Trôi
ICEBERG_THRESHOLD_PURCHASES = 500   # Ngưỡng số lượng đơn hàng tối thiểu trong ô
ICEBERG_THRESHOLD_REVENUE = 10000   # Ngưỡng doanh thu tối thiểu trong ô (USD)

print(f"[*] Quét Base Cuboid với điều kiện biên: Purchases >= {ICEBERG_THRESHOLD_PURCHASES} & Revenue >= {ICEBERG_THRESHOLD_REVENUE} USD")

# 2. Áp dụng mệnh đề lọc Iceberg Cube trên khối nền
iceberg_cube = data_cube[
    (data_cube['total_purchases'] >= ICEBERG_THRESHOLD_PURCHASES) & 
    (data_cube['total_revenue'] >= ICEBERG_THRESHOLD_REVENUE)
].copy()

# 3. Ghi nhận dữ liệu vật lý phần nổi của tảng băng
iceberg_cube.to_csv(PATH_ICEBERG, index=False)
print(f"-> Trích xuất thành công Iceberg Cube!")
print(f"   + Số lượng phần tử khối nền: {len(data_cube)} cụm.")
print(f"   + Số lượng phần tử nổi (Thỏa ngưỡng): {len(iceberg_cube)} cụm.")

print("\nTop 5 cụm Dimension cốt lõi dẫn đầu doanh số trong Iceberg Cube:")
print(iceberg_cube.sort_values(by='total_revenue', ascending=False).head(5))

=== THỰC THI THUẬT TOÁN ICEBERG CUBE TỐI ƯU ===
[*] Quét Base Cuboid với điều kiện biên: Purchases >= 500 & Revenue >= 10000 USD
-> Trích xuất thành công Iceberg Cube!
   + Số lượng phần tử khối nền: 573914 cụm.
   + Số lượng phần tử nổi (Thỏa ngưỡng): 222 cụm.

Top 5 cụm Dimension cốt lõi dẫn đầu doanh số trong Iceberg Cube:
       brand           category_code day_of_week  hour  total_interactions  \
43947  apple  electronics.smartphone   Wednesday     9               31086   
43923  apple  electronics.smartphone     Tuesday     9               31442   
43898  apple  electronics.smartphone    Thursday     8               29608   
43921  apple  electronics.smartphone     Tuesday     7               30377   
43922  apple  electronics.smartphone     Tuesday     8               30323   

       total_purchases  total_revenue  
43947             1591     1393887.78  
43923             1533     1332871.62  
43898             1439     1267793.45  
43921             1407     1246109.53  
439

Khai phá Tập phổ biến & Luật kết hợp (Frequent Pattern Mining)

In [7]:
print("=== KHAI PHÁ DỮ LIỆU: THUẬT TOÁN FP-GROWTH ===")

# 1. Trích xuất phân hệ ma trận giao dịch giỏ hàng (Market Basket Analysis)

print("[Step 1] Đang biến đổi dữ liệu sang dạng ma trận nhị phân giao dịch (One-hot status)...")
basket_matrix = (df_basket_source.groupby(['user_session', 'brand'])['brand']
                 .size().unstack().reset_index().fillna(0)
                 .set_index('user_session'))
basket_matrix = basket_matrix.map(lambda x: True if x > 0 else False)

# 2. Khai phá tập phổ biến với ngưỡng Support tối thiểu 1%
print("[Step 2] Đang quét tìm các tập thương hiệu phổ biến (min_support = 0.01)...")
frequent_patterns = fpgrowth(basket_matrix, min_support=0.01, use_colnames=True)
print(f" -> Tìm thấy {len(frequent_patterns)} tập thương hiệu phổ biến thỏa mãn.")

# 3. Sinh hệ thống luật kết hợp từ tập phổ biến
if len(frequent_patterns) > 0:
    print("[Step 3] Đang tính toán hệ thống luật kết hợp mạnh (min_confidence = 0.1)...")
    association_rules_df = association_rules(frequent_patterns, metric="confidence", min_threshold=0.1)
    association_rules_df = association_rules_df.sort_values(by=['lift', 'confidence'], ascending=False)
    
    # Xuất dữ liệu luật ra file báo cáo
    association_rules_df.to_csv(PATH_RULES, index=False)
    print(f" -> Khai phá hoàn tất! Đã lưu hệ thống gồm {len(association_rules_df)} luật kết hợp tại: {PATH_RULES}")
    print(association_rules_df[['antecedents', 'consequents', 'support', 'confidence', 'lift']].head(5))
else:
    print("[!] Không tìm thấy luật kết hợp phù hợp. Cần cân nhắc hạ ngưỡng min_support ở chu kỳ sau.")

=== KHAI PHÁ DỮ LIỆU: THUẬT TOÁN FP-GROWTH ===
[Step 1] Đang biến đổi dữ liệu sang dạng ma trận nhị phân giao dịch (One-hot status)...


AttributeError: 'DataFrame' object has no attribute 'map'

Tối ưu bộ nhớ & Giải phóng RAM (Memory Cleanup)

In [ ]:
print("=== KẾT THÚC CHU TRÌNH PHÂN TÍCH FILE HIỆN TẠI ===")

# Thực hiện giải phóng bộ nhớ chủ động
variables_to_clear = ['df', 'data_cube', 'slice_df', 'dice_df', 'rollup_df', 
                      'pivot_matrix', 'iceberg_cube', 'df_basket_source', 
                      'basket_matrix', 'frequent_patterns', 'association_rules_df']

for var in variables_to_clear:
    if var in locals() or var in globals():
        exec(f"del {var}")

# Gọi bộ gom rác thu hồi RAM lập tức
gc.collect()
print("[✔] ĐÃ GIẢI PHÓNG RAM SẠCH SẼ!")
print("[>] HƯỚNG DẪN: Quay lại Cell 1, thay đổi 'CURRENT_FILE_INDEX = 1' và chạy lại từ đầu để xử lý file kế tiếp.")